In [17]:
import torch
from torch import nn
import wandb
from sklearn.metrics import f1_score
from collections import Counter
import nltk
from nltk.corpus import stopwords

In [18]:
nltk.download('stopwords')
stop_words = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
wandb.init()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: theotheo46 (theotheo46-trs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
class RNN(nn.Module):
    def __init__(self, vocab_size, input_size=128, hidden_size=128, n_classes=9):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, input_size)

        self.hidden_size = hidden_size
        self.hidden_fc = nn.Linear(input_size + hidden_size, hidden_size)
        self.out_fc = nn.Linear(hidden_size, n_classes)

    def forward(self, input_ids, init_h=None):
        bs, seq_len = input_ids.shape

        x = self.embedding(input_ids)

        if init_h is None:
            # инициализируем скрытое состояние нулями
            h_t = torch.zeros(bs, self.hidden_size, device=x.device)
        else:
            h_t = init_h

        hidden_states = []
        for t in range(seq_len):
            # обновляем скрытое состояние RNN и сохраняем его в массив
            x_t = x[:, t, :]
            cat_state = torch.cat((x_t, h_t), dim=1)
            h_t = torch.tanh(self.hidden_fc(cat_state))

            hidden_states.append(h_t.unsqueeze(1))

        # применяем линейный слой ко всем скрытым состояниям, чтобы получить логиты
        hidden_states = torch.cat(hidden_states, dim=1)
        out_hidden_states = self.out_fc(hidden_states)

        return out_hidden_states


In [19]:
def process_datasets(texts_train, texts_test, min_wf, max_wf):
    word_counter = Counter()
    for text in texts_train:
        word_counter.update(Counter(text))

    def process_text(text):
        clean_text = []
        for word in text:
            if word not in stop_words and word in word_counter and min_wf <= word_counter[word] <= max_wf:
                clean_text.append(word)

        return clean_text

    texts_train = [process_text(text) for text in texts_train]
    texts_test = [process_text(text) for text in texts_test]

    return texts_train, texts_test

In [7]:
def compute_f1(y_true, y_pred):
    assert y_true.ndim == 2
    assert y_true.shape == y_pred.shape

    return f1_score(y_true, y_pred, average='macro')

In [8]:
def train(model, dataloader, optimizer, device='cpu', logging=False):
    """
    Обучает модель (model) на всем наборе данных (dataloader).
    """
    model.to(device)
    model.train()
    criterion = nn.BCELoss()

    for input_ids, labels in dataloader:
        preds = model(input_ids.to(device))
        loss = criterion(preds, labels.to(device))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        f1 = compute_f1((preds > 0.5).int().cpu(), labels)

        # логируем значения ошибки и f1
        if logging:
            wandb.log({
                "train_loss": loss.item(),
                "train_f1": f1
            })

In [9]:
@torch.inference_mode()
def evaluate(model, dataloader, device='cpu'):
    """
    Тестирует модель (model) на всем наборе данных (dataloader).
    """

    # не забываем переводить в eval режим
    model.to(device)
    model.eval()

    all_predictions = []
    all_labels = []
    for input_ids, labels in dataloader:
        preds = model(input_ids.to(device))

        all_predictions.extend((preds > 0.5).int().cpu())
        all_labels.extend(labels.cpu())

    # усредняем в самом конце, чтобы не зависеть от размера батча
    f1 = compute_f1(torch.stack(all_predictions), torch.stack(all_labels))

    return f1